# Seen and Unseen — Experiment 3 pilots on an A100: architecture axis and dose-response axis (effect-blind)

Both parts use the registered 2D reuse design unchanged (same events, appearances, arms, steps, contrasts) and release only dispersions, competence counts, runtime and the pooled coarse positive-control flag. **No effect location is written anywhere.**

| Part | What changes vs the registered design | Blocks |
|---|---|---|
| A — architecture | Predictor = frozen pretrained encoder (DINOv2-small 22 M, or V-JEPA 2-L 326 M; never trained) + trained latent transformer (width 256, depth 4; 3.3 M) instead of the ConvLSTM | 24 per encoder |
| B — dose | ConvLSTM-16 with 16 or 256 reused source trajectories instead of 64, so each reused trajectory is presented ~50 or ~3 times instead of ~12 | 24 per dose |

**Speed:** blocks run `WORKERS` at a time in separate processes on the one GPU (results do not depend on the worker count; checked), the frozen encoders run in bf16, and every finished block is checkpointed (blinded) on Drive. **After any disconnect: Runtime → Run all**; finished blocks are skipped.

**Runtime (A100, WORKERS = 3):** about 2–3 h for everything; Part A alone about 1–1.5 h. **Send back:** `exp3_pilots_results.zip`.

In [ ]:
#@title 0. Configuration
WORKERS = 3                    #@param {type:"integer"}   # blocks in parallel on the GPU; 3 on an A100 (40 GB), 2 on an L4
RUN_TUNING = True              #@param {type:"boolean"}
RUN_DINOV2 = True              #@param {type:"boolean"}
RUN_VJEPA = True               #@param {type:"boolean"}
RUN_DOSE = True                #@param {type:"boolean"}
TUNE_BLOCKS = 4                #@param {type:"integer"}
TUNE_CHECKPOINTS = "300,600,1200"  #@param {type:"string"}
EXP3_DRIVE_DIR = "/content/drive/MyDrive/seen-and-unseen/exp3/pilots_oct28"  #@param {type:"string"}
import os, torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (switch to a GPU runtime)", "| CPUs:", os.cpu_count(), "| workers:", WORKERS)
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(f"{EXP3_DRIVE_DIR}/DO_NOT_OPEN", exist_ok=True)
PILOTS = {  # name -> (config, run flag)
    "latent_dinov2": ("experiments/exposure2d_latent_dinov2_pilot_v1.json", RUN_DINOV2),
    "latent_vjepa2": ("experiments/exposure2d_latent_vjepa2_pilot_v1.json", RUN_VJEPA),
    "dose16": ("experiments/exposure2d_dose16_pilot_v1.json", RUN_DOSE),
    "dose256": ("experiments/exposure2d_dose256_pilot_v1.json", RUN_DOSE),
}
for k in PILOTS:
    ck = f"{EXP3_DRIVE_DIR}/{k}_blinded_checkpoint.jsonl"
    print(f"{k:14s} blocks already recorded: {sum(1 for _ in open(ck)) if os.path.exists(ck) else 0}")

In [ ]:
assert "WORKERS" in globals(), "Run the configuration cell at the top first (after any restart: Runtime -> Run all)"
#@title 1. Code: public trajectory-lineage at 6b1031b10ac5ec757e8bd8e7f1a2b47f0ddcbfef (includes pull request #19)
%cd /content
import os
if not os.path.exists("trajectory-lineage"):
    !git clone -q https://github.com/physicalreasoning/trajectory-lineage.git
%cd /content/trajectory-lineage
!git checkout -q 6b1031b10ac5ec757e8bd8e7f1a2b47f0ddcbfef && git rev-parse --short HEAD
!pip install -q numpy h5py pytest "transformers>=5.0" accelerate
print("ok")

In [ ]:
assert "WORKERS" in globals(), "Run the configuration cell at the top first (after any restart: Runtime -> Run all)"
#@title 2. Tests (CPU, about 1 min); load the encoders once so the weights are cached
%cd /content/trajectory-lineage
!git log --oneline -1
!PYTHONPATH=src python -m pytest -q tests/test_latent.py tests/test_capacity.py -p no:cacheprovider 2>&1 | tail -n 2
import torch, sys
sys.path.insert(0, "/content/trajectory-lineage/src")
from trajectory_lineage.latent import FrozenEncoder
for name, flag in (("dinov2-small", RUN_DINOV2), ("vjepa2-vitl", RUN_VJEPA)):
    if flag:
        enc = FrozenEncoder(name, torch.device("cuda"))
        tok = enc.encode(torch.rand(4, 8, 3, 64, 64))
        print(f"{name}: {enc.hub_id}, frozen params {enc.parameters_frozen/1e6:.0f}M, tokens {tuple(tok.shape)}")

In [ ]:
assert "WORKERS" in globals(), "Run the configuration cell at the top first (after any restart: Runtime -> Run all)"
#@title 3. Effect-blind base tuning for the latent predictor (base models only; no arms)
%cd /content/trajectory-lineage
import os
for k in ("latent_dinov2", "latent_vjepa2"):
    cfg, flag = PILOTS[k]
    out = f"results/{k}_base_tuning.json"
    if RUN_TUNING and flag and not os.path.exists(f"{EXP3_DRIVE_DIR}/{k}_base_tuning.json"):
        print(f"=== tuning {k} ===")
        !PYTHONPATH=src python scripts/tune_base_2d.py --config {cfg} --out {out} --blocks {TUNE_BLOCKS} --checkpoints {TUNE_CHECKPOINTS} --hidden 256 2>&1 | grep -v -i "warn\|Loading weights" | tail -n 8
        !cp {out} {EXP3_DRIVE_DIR}/{k}_base_tuning.json
    else:
        print(f"tuning {k}: skipped (disabled or already on Drive)")

In [ ]:
assert "WORKERS" in globals(), "Run the configuration cell at the top first (after any restart: Runtime -> Run all)"
#@title 4. Blind pilots, WORKERS blocks at a time, resumable from Drive; one line per finished block with an ETA
%cd /content/trajectory-lineage
import os
for k, (cfg, flag) in PILOTS.items():
    if not flag:
        print(f"{k}: disabled"); continue
    ck = f"{EXP3_DRIVE_DIR}/{k}_blinded_checkpoint.jsonl"; key = f"{EXP3_DRIVE_DIR}/DO_NOT_OPEN/{k}_blinding_key.json"
    out = f"results/{k}_pilot_v1_blinded.json"
    print(f"=== pilot {k} ===")
    !PYTHONPATH=src python scripts/exposure_experiment.py --config {cfg} --out {out} --device cuda --workers {WORKERS} --blinded-checkpoint {ck} --blinding-key {key} 2>&1 | grep -E "^block|^resuming|Error|error"
    !cp {out} {EXP3_DRIVE_DIR}/{k}_pilot_v1_blinded.json

In [ ]:
assert "WORKERS" in globals(), "Run the configuration cell at the top first (after any restart: Runtime -> Run all)"
#@title 5. Blinded summary and download
import json, os
for k in PILOTS:
    path = f"{EXP3_DRIVE_DIR}/{k}_pilot_v1_blinded.json"
    if not os.path.exists(path):
        print(f"{k:14s} no report"); continue
    r = json.load(open(path))
    d = {name: round(v["standard_deviation_upper_limit"], 5) for name, v in r["dispersion"].items() if name in ("C_source_gain", "C_independent_harm", "C_reuse_gap", "C_minus_D_gap", "E_exact_gain")}
    print(f"{k:14s} completed {r['blocks_completed']}/{r['blocks_scheduled']}, competent {r['blocks_base_competent']}, positive control {r.get('positive_control_passed')}, "
          f"failures {len(r['infrastructure_failures'])}+{len(r['algorithmic_failures'])}, s/block {r['runtime']['seconds_per_block_mean']:.0f}, SD80 {d}")
!git -C /content/trajectory-lineage rev-parse HEAD > {EXP3_DRIVE_DIR}/code_commit.txt
!cd {EXP3_DRIVE_DIR} && zip -q -j /content/exp3_pilots_results.zip *.json code_commit.txt
!ls -la /content/exp3_pilots_results.zip
from google.colab import files
files.download("/content/exp3_pilots_results.zip")

### Notes
- Drive checkpoints hold blinded records only (contrasts shifted by the key in `DO_NOT_OPEN/`); never open that folder. Do not add `--unblind` or `--checkpoint`.
- If the GPU runs out of memory with V-JEPA (unlikely on an A100), set `WORKERS = 2` and Run all; finished blocks are kept.
- A level whose base models fail the competence screen is reported as untestable and is not registered; the others stand.